# Setup
**Make sure to read the instructions carefully!**

If you have other resources used in the Blender project and chose to *make all paths relative*, pack all of them into a zip archive. Alternatively, you can *pack all external file*.

* `blender_version` : Version of blender used to render the scene. You may define your own blender version.
* `blend_file_path` : Path to the blend file after unpacking the zip archive. If blend file is used, this is automatically ignored.
___
* `upload_type` : Select the type of upload method. `gdrive_relative` pulls everything from the folder specified.
* `drive_path` : Path to your blend/zip file relative to the root of your Google Drive if `google_drive` is selected. Must  state the file and its extension (.zip/.blend) **unless** `gdrive_relative` is selected.
* `url_blend` : Specify the URL to the blend/zip file if `url` is selected.
___
* `animation` : Specify whether animation or still image is rendered. If **still image** is used, put the frame number in `start_frame`.
* `start_frame, end_frame` : Specify the start and end frame for animation. You may put same value such as zero for both input to set the default frame in the blend file.
___
* `download_type` : Select the type of download method. `gdrive_direct` enables the frames to be outputted directly to Google Drive (zipping will be disabled).
* `output_name` : Name of the output frames, **do NOT include .blend!** (## for frame number)
* `zip_files` : Archive multiple animation frames automatically into a zip file.
* `drive_output_path` : Path to your frames/zip file in Google Drive.
___
* `gpu_enabled, cpu_enabled` : Toggle GPU and CPU for rendering. CPU might give a slight boost in rendering time but may varies depend on the project.
* `optix_enable` : Enable OptiX which may boost performance, may be incompatible depending on the version of blender, project and GPU allocated

After you are done, go to Runtime > Run All (Ctrl + F9) and upload your files or have Google Drive authorised below. See the [GitHub repo](https://github.com/syn73/blender-colab) for more information.

In [ ]:
blender_version = '5.2.3' #@param ['2.79b', '2.83.9', '2.93.18', '3.3.21', '3.6.23', '4.2.23', '4.5.14', '5.2.2', '5.2.3'] {allow-input: true}
blend_file_path = '.' #@param {type: 'string'}
#@markdown ---
upload_type = 'google_drive' #@param ['direct', 'google_drive', 'url', 'gdrive_relative'] {allow-input: false}
drive_path = 'blend/cake_upload1.blend' #@param {type: 'string'}
url_blend = 'https://drive.google.com/file/d/17R6e-udrTIkmNAijnYiWEb2m0p2v_3V8/view?usp=sharing' #@param {type: 'string'}
#@markdown ---
animation = False #@param {type: 'boolean'}
start_frame =  1#@param {type: 'integer'}
end_frame =  250#@param {type: 'integer'}
#@markdown ---
download_type = 'google_drive' #@param ['direct', 'google_drive', 'gdrive_direct'] {allow-input: false}
output_name = 'blender-##' #@param {type: 'string'}
zip_files = True #@param {type: 'boolean'}
drive_output_path = 'blender/output' #@param {type: 'string'}
#@markdown ---
gpu_enabled = True #@param {type:"boolean"}
optix_enabled = False #@param {type:"boolean"}
cpu_enabled = False #@param {type:"boolean"}

In [ ]:
import subprocess
import os
import sys

%cd /content

# Force standard sub-process execution with a fully cleared LD_PRELOAD environment
clean_env = os.environ.copy()
clean_env["LD_PRELOAD"] = ""

try:
    # Query nvidia-smi safely without hitting shell pollution
    gpu_output = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=gpu_name", "--format=csv,noheader"],
        env=clean_env,
        universal_newlines=True
    ).strip()

    # Store it back in the list format the rest of your notebook expects
    gpu = [gpu_output]
    print("Current GPU: " + gpu[0])

except Exception as e:
    print("Error querying GPU, fallback method applied...")
    # Clean fallback wrapper for the system shell
    gpu = !env -u LD_PRELOAD nvidia-smi --query-gpu=gpu_name --format=csv,noheader
    print("Current GPU: " + gpu[0])

# Keep your notebook's existing validation logic intact
if gpu[0] == "Tesla K80" and optix_enabled:
  print("OptiX disabled because of unsupported GPU")
  optix_enabled = False


In [ ]:
import os
import glob

# 1. Clear out the invalid path
os.environ["LD_PRELOAD"] = ""

# 2. Reinstall the updated library package
!apt-get remove -y libtcmalloc-minimal4t64
!apt-get install -y libtcmalloc-minimal4t64

# 3. Locate the actual path on your active system dynamically
actual_paths = glob.glob("/usr/lib/x86_64-linux-gnu/libtcmalloc_minimal.so.4*")

if actual_paths:
    # Safely bind the exact available path string
    os.environ["LD_PRELOAD"] = actual_paths[0]
    print(f"Successfully bound LD_PRELOAD to: {actual_paths[0]}")
else:
    print("Could not locate a valid libtcmalloc library path automatically.")


In [ ]:

import os
import shutil
import subprocess
from pathlib import Path
from urllib.parse import urlparse
from google.colab import files, drive

uploaded_filename = ""

# Mount Google Drive when needed.
if (
    upload_type in ["google_drive", "gdrive_relative"]
    or download_type in ["google_drive", "gdrive_direct"]
):
    if not Path("/content/drive/MyDrive").exists():
        if not Path("/drive/MyDrive").exists():
            drive.mount("/content/drive")

    mount_base = (
        Path("/content/drive/MyDrive")
        if Path("/content/drive/MyDrive").exists()
        else Path("/drive/MyDrive")
    )


def prepare_blender_file(source_file):
    """Prepare a .blend file or Blender backup for rendering."""
    source_file = Path(source_file).resolve()

    if not source_file.is_file():
        raise FileNotFoundError(f"File not found: {source_file}")

    name = source_file.name

    # Accept .blend, .blend1, .blend11, .blend111, etc.
    if not name.lower().endswith(".blend") and not (
        ".blend" in name.lower()
        and name.lower().split(".blend", 1)[1].isdigit()
    ):
        raise ValueError(f"Unsupported file type: {name}")

    if name.lower().endswith(".blend"):
        uploaded_filename = name
        print(f"Using Blender project: {source_file}")
        return str(source_file)

    # Keep the original backup unchanged.
    # Make a temporary .blend copy in /content for Blender.
    backup_suffix = name.lower().split(".blend", 1)[1]
    original_stem = name[:name.lower().index(".blend")]
    restored_name = f"{original_stem}.blend"
    restored_path = Path("/content") / restored_name

    shutil.copy2(source_file, restored_path)

    print(f"Blender backup detected: {name}")
    print(f"Created temporary render copy: {restored_path}")
    print("The original backup file was not changed.")

    return str(restored_path)


# Upload directly from your computer.
if upload_type == "direct":
    uploaded = files.upload()

    if not uploaded:
        raise RuntimeError("No file was uploaded.")

    selected_name = next(iter(uploaded))
    uploaded_filename = prepare_blender_file(selected_name)

# Download from a URL.
elif upload_type == "url":
    if not url_blend:
        raise ValueError("url_blend is empty.")

    subprocess.run(
        ["wget", "-nc", url_blend],
        check=True
    )

    downloaded_file = Path(urlparse(url_blend).path).name
    uploaded_filename = prepare_blender_file(downloaded_file)

# Copy a file from Google Drive.
elif upload_type == "google_drive":
    relative_path = Path(drive_path.lstrip("/"))

    if not drive_path or ".." in relative_path.parts:
        raise ValueError("Provide a valid file path inside MyDrive.")

    source_file = (mount_base / relative_path).resolve()

    if not source_file.is_relative_to(mount_base.resolve()):
        raise ValueError("The file must be inside MyDrive.")

    if not source_file.is_file():
        raise FileNotFoundError(f"Drive file not found: {source_file}")

    uploaded_filename = prepare_blender_file(source_file)

elif upload_type == "gdrive_relative":
    # Leave this mode to your separate staging cell.
    print("Google Drive relative upload will be handled by the staging cell.")

else:
    raise ValueError(f"Unsupported upload_type: {upload_type}")


In [ ]:

import os
import shutil
import zipfile
from pathlib import Path, PurePosixPath


# 1. Prepare the staging directory


os.chdir("/content")

render_dir = Path("/content/render")
render_dir.mkdir(parents=True, exist_ok=True)

# Remove only previous staged project files.
shutil.rmtree(render_dir)
render_dir.mkdir(parents=True, exist_ok=True)



# 2. Safe ZIP extraction


def extract_zip_safely(zip_path, destination):
    """Extract ZIP files while preventing path traversal."""

    destination = destination.resolve()

    with zipfile.ZipFile(zip_path, "r") as archive:
        for member in archive.infolist():
            member_path = PurePosixPath(member.filename)

            if (
                member_path.is_absolute()
                or ".." in member_path.parts
                or "\\" in member.filename
            ):
                raise ValueError(
                    f"Unsafe ZIP entry: {member.filename}"
                )

            target = (destination / Path(*member_path.parts)).resolve()

            if not target.is_relative_to(destination):
                raise ValueError(
                    f"ZIP entry escapes render directory: "
                    f"{member.filename}"
                )

            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue

            # Reject symbolic links in ZIP archives.
            unix_mode = (member.external_attr >> 16) & 0o170000

            if unix_mode == 0o120000:
                raise ValueError(
                    f"Symbolic links are not allowed: "
                    f"{member.filename}"
                )

            target.parent.mkdir(parents=True, exist_ok=True)

            with archive.open(member, "r") as source:
                with open(target, "wb") as output:
                    shutil.copyfileobj(
                        source,
                        output,
                        length=8 * 1024 * 1024,
                    )



# 3. Stage files according to upload mode


if upload_type == "gdrive_relative":

    # Google Drive must already be mounted.
    drive_base = Path("/content/drive/MyDrive").resolve()

    if not drive_base.is_dir():
        raise FileNotFoundError(
            "Google Drive is not mounted at "
            "/content/drive/MyDrive."
        )

    # drive_path is relative to MyDrive.
    relative_drive_path = Path(drive_path.strip("/"))

    if (
        relative_drive_path.is_absolute()
        or ".." in relative_drive_path.parts
    ):
        raise ValueError(
            "drive_path must be a relative path inside MyDrive."
        )

    source_dir = (
        drive_base / relative_drive_path
    ).resolve()

    if not source_dir.is_relative_to(drive_base):
        raise ValueError(
            "Google Drive source must remain inside MyDrive."
        )

    if not source_dir.is_dir():
        raise FileNotFoundError(
            f"Google Drive folder not found: {source_dir}"
        )

    print(f"📂 Copying project files from: {source_dir}")

    # Copy folder contents, including subdirectories.
    shutil.copytree(
        source_dir,
        render_dir,
        dirs_exist_ok=True,
    )


elif uploaded_filename.lower().endswith(".zip"):

    zip_path = Path(uploaded_filename).expanduser().resolve()

    if not zip_path.is_file():
        raise FileNotFoundError(
            f"Uploaded ZIP file not found: {zip_path}"
        )

    print(f"📦 Extracting: {zip_path.name}")

    extract_zip_safely(zip_path, render_dir)


elif uploaded_filename.lower().endswith(".blend"):

    blend_source = Path(
        uploaded_filename
    ).expanduser().resolve()

    if not blend_source.is_file():
        raise FileNotFoundError(
            f"Uploaded Blender file not found: {blend_source}"
        )

    staged_blend = render_dir / blend_source.name

    shutil.copy2(blend_source, staged_blend)

    print(f"📄 Staged Blender file: {staged_blend.name}")


else:
    raise ValueError(
        "Invalid upload type. Use a Google Drive folder, "
        ".zip archive, or .blend file."
    )



# 4. Find the staged Blender project


blend_files = sorted(
    render_dir.rglob("*.blend"),
    key=lambda path: str(path).lower(),
)

# Handle uppercase extensions such as .BLEND too.
blend_files += sorted(
    [
        path
        for path in render_dir.rglob("*")
        if path.is_file()
        and path.suffix.lower() == ".blend"
        and path not in blend_files
    ],
    key=lambda path: str(path).lower(),
)

if not blend_files:
    raise FileNotFoundError(
        "No .blend file was found in the staged render directory. "
        "Check your upload or Google Drive folder."
    )

if len(blend_files) > 1:
    print("⚠️ Multiple .blend files found:")

    for index, path in enumerate(blend_files, start=1):
        print(f"  {index}. {path.relative_to(render_dir)}")

    print(
        "Using the first file in sorted order. "
        "Change blend_file_path manually if needed."
    )

selected_blend = blend_files[0]

# Compatible with the earlier dynamic render cell.
blend_file_path = str(
    selected_blend.relative_to(Path("/content"))
)

print(f"\n✅ Project staged successfully.")
print(f"🎬 Selected blend file: {blend_file_path}")
print(f"📂 Staging directory: {render_dir}")


In [ ]:
import shutil
from pathlib import Path
import os

os.chdir("/content")

# Remove only known Blender installations and shortcuts.
for path in Path("/content").glob("blender-*"):
    if path.is_dir() and not path.is_symlink():
        shutil.rmtree(path)

blender_link = Path("/content/blender")

if blender_link.is_symlink():
    blender_link.unlink()

# Remove only if this is an intentional fresh Blender setup.
# Do not blindly delete /content/lib, /content/include,
# /content/share, or /usr/local/blender.

In [ ]:

import os
import re
import shutil
import tarfile
import tempfile
import requests
import subprocess

from pathlib import Path
from urllib.parse import urlparse


# 1. Configuration


blender_version = "5.2.3"

install_dir = Path("/content") / blender_version
target_binary = install_dir / "blender"
shortcut = Path("/content/blender")
archive_dir = Path("/content/blender_archives")

archive_dir.mkdir(parents=True, exist_ok=True)

# Custom URLs for older releases.
blender_url_dict = {
    "2.79b": (
        "https://download.blender.org/release/Blender2.79/"
        "blender-2.79b-linux-glibc219-x86_64.tar.bz2"
    ),
    "2.83.9": (
        "https://download.blender.org/release/Blender2.83/"
        "blender-2.83.9-linux64.tar.xz"
    ),
}


# 2. Validate version and construct the download URL


if not re.fullmatch(
    r"(?:\d+\.\d+\.\d+[a-z]?|\d+\.\d+[a-z])",
    blender_version,
):
    raise ValueError(f"Invalid Blender version: {blender_version!r}")

if blender_version in blender_url_dict:
    blender_url = blender_url_dict[blender_version]
else:
    parts = blender_version.split(".")
    major_minor = ".".join(parts[:2])

    blender_url = (
        "https://download.blender.org/release/"
        f"Blender{major_minor}/"
        f"blender-{blender_version}-linux-x64.tar.xz"
    )

parsed_url = urlparse(blender_url)

if parsed_url.scheme not in ("http", "https") or not parsed_url.netloc:
    raise ValueError(f"Invalid download URL: {blender_url}")

archive_name = os.path.basename(parsed_url.path)

if not archive_name:
    raise ValueError("The download URL has no archive filename.")

archive_path = archive_dir / archive_name
partial_path = archive_dir / (archive_name + ".part")

print(f"Requested Blender version: {blender_version}")
print(f"Download URL: {blender_url}")
print(f"Installation directory: {install_dir}")


# 3. Verify a Blender executable


def verify_blender(executable):
    """Check whether Blender launches and reports the expected version."""

    if not executable.is_file() or not os.access(executable, os.X_OK):
        return False

    try:
        result = subprocess.run(
            [str(executable), "--version"],
            capture_output=True,
            text=True,
            timeout=30,
            check=True,
        )

        output = result.stdout + "\n" + result.stderr

        match = re.search(
            r"^Blender\s+(\d+\.\d+(?:\.\d+)?[a-z]?)",
            output,
            re.MULTILINE,
        )

        if not match:
            print("Could not read Blender's reported version.")
            return False

        actual_version = match.group(1)

        # Blender 2.79b may report itself as Blender 2.79.
        version_matches = (
            actual_version == blender_version
            or (
                blender_version == "2.79b"
                and actual_version.startswith("2.79")
            )
        )

        if not version_matches:
            print(
                f"Version mismatch: expected {blender_version}, "
                f"found {actual_version}."
            )
            return False

        print(f"Verified Blender {actual_version}.")
        return True

    except (OSError, subprocess.SubprocessError) as error:
        print(f"Blender verification failed: {error}")
        return False



# 4. Reuse an existing installation or obtain the archive


if verify_blender(target_binary):
    print("Existing installation is usable. Skipping download.")

else:
    # Reuse the archive from your previous successful download.
    archive_is_valid = (
        archive_path.is_file()
        and tarfile.is_tarfile(archive_path)
    )

    if archive_is_valid:
        print(f"Reusing existing archive: {archive_path}")

    else:
        if archive_path.exists():
            archive_path.unlink()

        partial_path.unlink(missing_ok=True)

        print("Downloading Blender archive...")

        try:
            with requests.get(
                blender_url,
                stream=True,
                timeout=(20, 120),
                allow_redirects=True,
            ) as response:
                response.raise_for_status()

                content_length = response.headers.get("Content-Length")
                expected_size = (
                    int(content_length) if content_length else None
                )
                downloaded_size = 0

                with open(partial_path, "wb") as archive_file:
                    for chunk in response.iter_content(
                        chunk_size=8 * 1024 * 1024
                    ):
                        if chunk:
                            archive_file.write(chunk)
                            downloaded_size += len(chunk)

                if downloaded_size == 0:
                    raise RuntimeError("Downloaded archive is empty.")

                if (
                    expected_size is not None
                    and downloaded_size != expected_size
                ):
                    raise RuntimeError(
                        f"Incomplete download: expected {expected_size} "
                        f"bytes, received {downloaded_size} bytes."
                    )

            if not tarfile.is_tarfile(partial_path):
                raise RuntimeError(
                    "Downloaded file is not a valid TAR archive."
                )

            os.replace(partial_path, archive_path)

            print(
                f"Download completed: "
                f"{downloaded_size / (1024 ** 2):.1f} MiB"
            )

        except Exception as error:
            partial_path.unlink(missing_ok=True)
            raise RuntimeError(
                f"Blender download failed: {error}"
            ) from error

    # ========================================================
    # 5. Extract safely into a temporary staging directory
    # ========================================================

    staging_dir = Path(
        tempfile.mkdtemp(
            prefix="blender_install_",
            dir="/content",
        )
    )

    try:
        print("Extracting Blender archive...")

        with tarfile.open(archive_path, mode="r:*") as archive:
            members = archive.getmembers()

            if not members:
                raise RuntimeError("Archive contains no files.")

            # Expect one top-level folder in the official archive.
            top_levels = {
                member.name.split("/", 1)[0]
                for member in members
                if member.name and member.name != "."
            }

            if len(top_levels) != 1:
                raise RuntimeError(
                    "Unexpected archive structure: expected one "
                    "top-level directory."
                )

            # Python's data filter prevents unsafe extraction paths.
            archive.extractall(path=staging_dir, filter="data")

        extracted_root = staging_dir / next(iter(top_levels))
        staged_binary = extracted_root / "blender"

        if not verify_blender(staged_binary):
            raise RuntimeError(
                "Extracted Blender executable failed verification."
            )

        # Replace the old installation only after verifying the new one.
        if install_dir.is_symlink():
            install_dir.unlink()
        elif install_dir.exists():
            shutil.rmtree(install_dir)

        os.replace(extracted_root, install_dir)

        print("Blender extracted and installed.")

    except Exception as error:
        raise RuntimeError(
            f"Blender installation failed: {error}"
        ) from error

    finally:
        shutil.rmtree(staging_dir, ignore_errors=True)


# 6. Verify the final installation


if not verify_blender(target_binary):
    raise RuntimeError(
        f"Final installation is not usable: {target_binary}"
    )


# 7. Create or repair /content/blender shortcut


if shortcut.is_symlink():
    if shortcut.resolve() != install_dir.resolve():
        shortcut.unlink()
        shortcut.symlink_to(
            install_dir,
            target_is_directory=True,
        )

elif shortcut.exists():
    raise RuntimeError(
        f"{shortcut} already exists and is not a symlink. "
        "Inspect it before replacing it."
    )

else:
    shortcut.symlink_to(
        install_dir,
        target_is_directory=True,
    )

print(f"Blender shortcut: {shortcut}")
print(f"Actual executable: {target_binary}")
print("Blender installation is ready.")


In [ ]:
# Ensure variables are defined for the string format evaluation
gpu_enabled = True
cpu_enabled = False
optix_enabled = False  # Set to True if you want to use OptiX on your T4 later

data = "import re\n"+\
    "import bpy\n"+\
    "scene = bpy.context.scene\n"+\
    "scene.render.engine = 'CYCLES'\n"+\
    "scene.cycles.device = 'GPU'\n"+\
    "prefs = bpy.context.preferences\n"+\
    "prefs.addons['cycles'].preferences.get_devices()\n"+\
    "cprefs = prefs.addons['cycles'].preferences\n"+\
    "print(cprefs)\n"+\
    "for compute_device_type in ('OPTIX', 'CUDA', 'OPENCL', 'NONE'):\n"+\
    "    try:\n"+\
    "        cprefs.compute_device_type = compute_device_type\n"+\
    "        print('Device found:',compute_device_type)\n"+\
    "        break\n"+\
    "    except TypeError:\n"+\
    "        pass\n"+\
    "for device in cprefs.devices:\n"+\
    "    if not re.match('intel', device.name, re.I):\n"+\
    "        print('Activating',device.name)\n"+\
    "        device.use = "+str(gpu_enabled)+"\n"+\
    "    else:\n"+\
    "        device.use = "+str(cpu_enabled)+"\n"+\
    "bpy.ops.wm.save_userpref()\n"

with open('setgpu.py', 'w') as f:
    f.write(data)

renderer = "CUDA"
if optix_enabled:
    print("Note: You're currently using OptiX renderer. If an error occurred, the current GPU (e.g. Tesla K80) is not supported and you need to switch back to CUDA.")
    renderer = "OPTIX"

print("📝 'setgpu.py' has been successfully created.")
print("--------------------------------------------------")

# --- EXECUTE THE CONFIGURATION SCRIPT IN BLENDER ---
import os
if os.path.exists("/content/blender/blender"):
    get_ipython().system("/content/blender/blender -b -P setgpu.py")
else:
    print("❌ Error: Could not find your blender install folder shortcut at /content/blender.")


In [ ]:

import os
import re
import glob
import shutil
import subprocess
from pathlib import Path


# 1. Blender version configuration


# Leave empty for automatic detection.
# Example override: "5.2.3"
blender_version_override = ""

# Optional: explicitly specify the Blender executable if it
# is installed in a non-standard location.
# Example: "/content/blender-5.2.3-linux-x64/blender"
blender_executable_override = ""

gpu_script = Path("/content/setgpu.py")



# 2. Detect installed Blender versions


def get_blender_version(executable):
    """Return the installed Blender version as a tuple."""

    try:
        result = subprocess.run(
            [str(executable), "--version"],
            capture_output=True,
            text=True,
            timeout=30,
            check=True,
        )

        match = re.search(
            r"^Blender\s+(\d+)\.(\d+)\.(\d+)",
            result.stdout,
            re.MULTILINE,
        )

        if match:
            return tuple(map(int, match.groups()))

    except (
        OSError,
        subprocess.SubprocessError,
    ):
        return None

    return None


def find_blender_installations():
    """Find Blender executables in common Colab locations."""

    candidates = set()

    # Common installation layouts.
    patterns = [
        "/content/*/blender",
        "/content/blender",
        "/usr/bin/blender",
        "/usr/local/bin/blender",
        "/usr/local/blender/*/blender",
    ]

    for pattern in patterns:
        for candidate in glob.glob(pattern):
            if os.path.isfile(candidate) and os.access(
                candidate, os.X_OK
            ):
                candidates.add(os.path.abspath(candidate))

    # Also consider Blender available on PATH.
    path_blender = shutil.which("blender")

    if path_blender:
        candidates.add(os.path.abspath(path_blender))

    installations = []

    for candidate in candidates:
        version = get_blender_version(candidate)

        if version is not None:
            installations.append((version, candidate))

    # Sort numerically: major, minor, then patch.
    installations.sort(key=lambda item: item[0], reverse=True)

    return installations


installations = find_blender_installations()


# 3. Select Blender version


if blender_executable_override:
    # An explicitly supplied executable takes precedence.
    blender_bin = os.path.abspath(blender_executable_override)

    if not os.path.isfile(blender_bin):
        raise FileNotFoundError(
            f"Blender executable not found: {blender_bin}"
        )

    if not os.access(blender_bin, os.X_OK):
        raise PermissionError(
            f"Blender executable is not executable: {blender_bin}"
        )

    selected_version = get_blender_version(blender_bin)

    if selected_version is None:
        raise RuntimeError(
            f"Could not identify Blender version at: {blender_bin}"
        )

    if blender_version_override:
        requested_version = tuple(
            map(int, blender_version_override.split("."))
        )

        if selected_version != requested_version:
            raise RuntimeError(
                f"Requested Blender {blender_version_override}, "
                f"but the executable is Blender "
                f"{'.'.join(map(str, selected_version))}."
            )

elif blender_version_override:
    # Match the explicitly requested major.minor.patch version.
    if not re.fullmatch(
        r"\d+\.\d+\.\d+", blender_version_override
    ):
        raise ValueError(
            "Version must use major.minor.patch format, "
            'for example "5.2.3".'
        )

    requested_version = tuple(
        map(int, blender_version_override.split("."))
    )

    matches = [
        (version, executable)
        for version, executable in installations
        if version == requested_version
    ]

    if not matches:
        found = [
            f"{'.'.join(map(str, version))}: {executable}"
            for version, executable in installations
        ]

        raise FileNotFoundError(
            f"Blender {blender_version_override} was not found.\n"
            f"Detected installations:\n"
            + ("\n".join(found) if found else "None")
            + "\nInstall the requested version or check its path."
        )

    selected_version, blender_bin = matches[0]

else:
    # Automatic selection: highest installed semantic version.
    if not installations:
        raise FileNotFoundError(
            "No usable Blender installation was detected.\n"
            "Install Blender, or set blender_executable_override "
            'to its executable path, e.g. "/content/5.2.3/blender".'
        )

    selected_version, blender_bin = installations[0]

version_text = ".".join(map(str, selected_version))

print(f"✅ Selected Blender version: {version_text}")
print(f"📍 Executable: {blender_bin}")

if len(installations) > 1 and not blender_version_override:
    print("Detected Blender installations:")

    for version, executable in installations:
        print(f"  {'.'.join(map(str, version))}: {executable}")



# 4. Prepare output directory and resolve paths


output_dir = Path("/content/output")

# Remove previous local render outputs.
if output_dir.exists():
    shutil.rmtree(output_dir)

output_dir.mkdir(parents=True, exist_ok=True)

if download_type == "gdrive_direct":
    drive_base = Path("/content/drive/MyDrive")

    if not drive_base.is_dir():
        raise FileNotFoundError(
            "Google Drive is not mounted at "
            "/content/drive/MyDrive."
        )

    drive_dir = Path(drive_output_path.strip("/"))

    if drive_dir.is_absolute() or ".." in drive_dir.parts:
        raise ValueError(
            "drive_output_path must be a relative path inside MyDrive."
        )

    drive_target = (drive_base / drive_dir).resolve()

    if not drive_target.is_relative_to(drive_base.resolve()):
        raise ValueError(
            "Google Drive destination must remain inside MyDrive."
        )

    drive_target.mkdir(parents=True, exist_ok=True)
    output_path = str(drive_target / output_name)

else:
    output_path = str(output_dir / output_name)


# Resolve the .blend file.
blend_input = Path(blend_file_path)

if blend_input.is_absolute():
    blend_file = blend_input

elif blend_file_path.startswith("./render/"):
    blend_file = Path("/content") / blend_input

elif blend_file_path.startswith("render/"):
    blend_file = Path("/content") / blend_input

else:
    render_candidate = Path("/content/render") / blend_input
    content_candidate = Path("/content") / blend_input

    if render_candidate.is_file():
        blend_file = render_candidate
    else:
        blend_file = content_candidate

blend_file = blend_file.resolve()

if not blend_file.is_file():
    raise FileNotFoundError(
        f"Blend file not found: {blend_file}"
    )

if not gpu_script.is_file():
    raise FileNotFoundError(
        f"GPU setup script not found: {gpu_script}"
    )

print(f"🎬 Blend file: {blend_file}")
print(f"📂 Output prefix: {output_path}")
print(f"🖥️ Cycles device: {renderer}")



# 5. Validate render settings


if not isinstance(start_frame, int) or not isinstance(end_frame, int):
    raise TypeError("start_frame and end_frame must be integers.")

if start_frame < 0 or end_frame < 0:
    raise ValueError("Frame numbers cannot be negative.")

if start_frame > end_frame:
    raise ValueError(
        "start_frame cannot be greater than end_frame."
    )

# Keep the renderer argument as one shell-free argument.
# Common Cycles device choices: CPU, CUDA, OPTIX, HIP, ONEAPI, METAL.
renderer = str(renderer).strip().upper()

allowed_renderers = {
    "CPU", "CUDA", "OPTIX", "HIP", "ONEAPI", "METAL"
}

if renderer not in allowed_renderers:
    raise ValueError(
        f"Unsupported renderer: {renderer!r}. "
        f"Choose one of: {', '.join(sorted(allowed_renderers))}."
    )



# 6. Run Blender and show live render output


from IPython.display import display, clear_output, Image
import subprocess
import time

command = [
    blender_bin,
    "-b", str(blend_file),
    "-P", str(gpu_script),
    "-E", "CYCLES",
    "-o", output_path,
    "-noaudio",
]

if animation:
    command.extend([
        "-s", str(start_frame),
        "-e", str(end_frame),
        "-a",
    ])
else:
    command.extend(["-f", str(start_frame)])

command.extend(["--", "--cycles-device", renderer])

print("Starting Blender render...")
print("Blender:", blender_bin)
print("Project:", blend_file)
print("Output:", output_path)
print("Renderer:", renderer)
print()

# Stream Blender's output to the notebook.
process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in process.stdout:
        print(line, end="", flush=True)

    return_code = process.wait()

except KeyboardInterrupt:
    process.terminate()
    process.wait()
    raise RuntimeError("Render interrupted by user.")

if return_code != 0:
    raise RuntimeError(
        f"Blender render failed with exit code {return_code}."
    )

print("\nBlender render completed successfully.")


# 7. List and preview completed output files


output_directory = (
    drive_target if download_type == "gdrive_direct"
    else output_dir
)

generated_files = sorted(
    path for path in output_directory.iterdir()
    if path.is_file()
)

print(f"\nFound {len(generated_files)} output file(s).")

for path in generated_files:
    print(path.name)

    # Display rendered images directly in Colab.
    if path.suffix.lower() in [".png", ".jpg", ".jpeg", ".webp"]:
        display(Image(filename=str(path)))

print("\nRender routine complete.")


In [ ]:

import os
import re
import shutil
import zipfile
from google.colab import files

# 1. Configure paths
drive_base = "/content/drive/MyDrive"
output_dir = "/content/output"

output_folder_name = (
    output_name.replace("#", "").strip() + "_render"
)

os.makedirs(output_dir, exist_ok=True)

# 2. Sort filenames naturally
def natural_sort_key(filename):
    return [
        int(part) if part.isdigit() else part.lower()
        for part in re.split(r"(\d+)", filename)
    ]


# Only include regular files, sorted by filename
files_folder = sorted(
    [
        filename
        for filename in os.listdir(output_dir)
        if os.path.isfile(os.path.join(output_dir, filename))
    ],
    key=natural_sort_key,
)

print(
    f"📦 Post-Render Engine: Found "
    f"{len(files_folder)} file(s) in the output directory."
)

# 3. Configure large-file transfers
# Copies files in chunks without loading the whole file into memory.
def copy_large_file(source, destination):
    with open(source, "rb") as source_file:
        with open(destination, "wb") as destination_file:
            shutil.copyfileobj(
                source_file,
                destination_file,
                length=8 * 1024 * 1024,
            )


# Creates a ZIP archive without loading the entire render into RAM.
# ZIP_STORED avoids spending extra CPU compressing PNG/JPEG frames.
def create_render_archive(source_dir, archive_path, filenames):
    with zipfile.ZipFile(
        archive_path,
        mode="w",
        compression=zipfile.ZIP_STORED,
        allowZip64=True,
    ) as archive:
        for index, filename in enumerate(filenames, start=1):
            source_path = os.path.join(source_dir, filename)

            archive.write(source_path, arcname=filename)

            if index % 100 == 0 or index == len(filenames):
                print(
                    f"🗜️ Archived {index}/{len(filenames)} files"
                )


# 4. Route the rendered files
if download_type == "gdrive_direct":
    print(
        "✅ Render saved directly to Google Drive "
        "by the upstream rendering pipeline."
    )

elif not files_folder:
    print(
        "❌ No files found in the output directory. "
        "Check Blender's rendering logs."
    )

else:
    clean_drive_dir = drive_output_path.strip("/")

    target_drive_path = os.path.abspath(
        os.path.join(drive_base, clean_drive_dir)
    )

    # Prevent accidental copies outside MyDrive.
    if os.path.commonpath(
        [os.path.abspath(drive_base), target_drive_path]
    ) != os.path.abspath(drive_base):
        raise ValueError(
            "Google Drive destination must remain inside MyDrive."
        )

    # Create the Drive destination when copying files there.
    if download_type != "direct":
        os.makedirs(target_drive_path, exist_ok=True)

    # SCENARIO A: One file
    if len(files_folder) == 1:
        filename = files_folder[0]
        local_file_path = os.path.join(output_dir, filename)

        if download_type == "direct":
            print(f"📥 Downloading: {filename}")
            files.download(local_file_path)

        else:
            destination = os.path.join(
                target_drive_path, filename
            )

            print(f"💾 Copying to Google Drive: {filename}")
            copy_large_file(local_file_path, destination)

    # SCENARIO B: Multiple files
    elif len(files_folder) > 1:

        if zip_files:
            archive_name = output_folder_name + ".zip"
            archive_path = os.path.join("/content", archive_name)

            print(
                f"🗜️ Creating archive for "
                f"{len(files_folder)} files: {archive_name}"
            )

            create_render_archive(
                output_dir,
                archive_path,
                files_folder,
            )

            archive_size_gb = (
                os.path.getsize(archive_path) / (1024 ** 3)
            )

            print(
                f"✅ Archive created: {archive_size_gb:.2f} GB"
            )

            if download_type == "direct":
                print("📥 Starting browser download...")
                files.download(archive_path)

            else:
                destination = os.path.join(
                    target_drive_path, archive_name
                )

                print("💾 Copying archive to Google Drive...")
                copy_large_file(archive_path, destination)

                print(f"✅ Saved to: {destination}")

        else:
            # Transfer individual files in natural sort order.
            if download_type == "direct":
                print(
                    "📥 Downloading individual files. "
                    "Your browser may limit multiple downloads."
                )

                for index, filename in enumerate(
                    files_folder, start=1
                ):
                    print(
                        f"Downloading {index}/{len(files_folder)}: "
                        f"{filename}"
                    )
                    files.download(
                        os.path.join(output_dir, filename)
                    )

            else:
                print(
                    f"💾 Copying {len(files_folder)} files "
                    f"to Google Drive..."
                )

                for index, filename in enumerate(
                    files_folder, start=1
                ):
                    source = os.path.join(output_dir, filename)
                    destination = os.path.join(
                        target_drive_path, filename
                    )

                    copy_large_file(source, destination)

                    if index % 100 == 0 or index == len(files_folder):
                        print(
                            f"Copied {index}/{len(files_folder)} files"
                        )

                print(f"✅ Saved to: {target_drive_path}")

print("🏁 Post-processing routine complete.")


## Disclaimer
Google Colab is targeted to researchers and students to run AI/ML tasks, data analysis and education, not rendering 3D scenes. Because the computing power provided are free, the usage limits, idle timeouts and speed of the rendering may varies time by time. [Colab Pro and Colab Pro+](https://colab.research.google.com/signup) are available for those who wanted to have more powerful GPU and longer runtimes for rendering. See the [FAQ](https://research.google.com/colaboratory/faq.html) for more info. In some cases, it might be faster to use an online Blender renderfarm.

## License
```
MIT License

Copyright (c) 2020-2022 ynshung

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
```